# Stable Production — DistilBERT + Hybrid Ensemble

Production DistilBERT settings:
- **Layers:** freeze first 4 / train last 2 + head
- **Training:** up to 15 epochs, early stopping patience=3 on **val `f1_toxic`**
- **Regularization:** dropout 0.5, label smoothing 0.1, AdamW 1e-5

Ensemble: soft vote (0.5 BERT + 0.5 LR probabilities).

## 0. Load integrated metrics

In [ ]:
import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "reports").exists() and (PROJECT_ROOT.parent / "reports").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

reports_dir = PROJECT_ROOT / "reports" / "stable"
latest = sorted(reports_dir.glob("stable_run_*.json"))[-1]
metrics = json.loads(latest.read_text())
run_id = metrics["run_id"]
print(latest)

## 1. Holdout test — all models

In [ ]:
def _row(key, label):
    m = metrics.get(key, {})
    if not m:
        return None
    gap_pp = m.get("train_test_gap_pp", m.get("train_test_gap", 0) * 100)
    return {
        "model": label,
        "f1_test": m.get("f1_weighted"),
        "f1_toxic": m.get("f1_toxic"),
        "f1_train": m.get("f1_train"),
        "gap_pp": gap_pp,
        "gap_ok": gap_pp < 5,
        "roc_auc": m.get("roc_auc"),
        "fp": m.get("fp"),
        "fn": m.get("fn"),
    }

rows = [
    _row("distilbert", "DistilBERT"),
    _row("logistic_regression", "LR-TFIDF"),
    _row("ensemble", "Hybrid"),
]
summary = pd.DataFrame([r for r in rows if r])
summary

## 2. Integrated markdown report

In [ ]:
from IPython.display import Markdown, display

md_path = reports_dir / f"integrated_report_{run_id}.md"
if md_path.exists():
    display(Markdown(md_path.read_text()))
else:
    print("Report not found — re-run pipeline")

## Conclusion

DistilBERT is trained after LR passes gap search. **Weighted F1** can look low when the model favors recall on the toxic class (many false positives).
Inspect **`f1_toxic`**, ROC-AUC, and FP/FN alongside the train–test gap.
Artifacts: `models/stable_distilbert/`, `models/stable_lr_tfidf.joblib`, `models/stable_ensemble_meta.json`.